# STAG 数据集探索（与训练逻辑无关）

本 notebook 只帮助理解 `classification_lite.zip` 中的原始数据、字段、recording、时间连续性和 548 点传感器掩码，不导入 SNN 模型，也不执行训练。

重点问题：

1. `metadata.mat` 里实际有哪些字段？
2. 27 个原始类别和官方 train/test 是怎样分布的？
3. `32×32` 矩阵中哪些位置是真实传感器？
4. 一条 recording 的压力和 `hasValidLabel` 如何随时间变化？
5. 为什么 `isBalanced` 不能用于连续时序窗口？

In [1]:
%matplotlib inline

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 同时兼容“从 code 目录启动”和“从项目根目录启动”两种方式。
CODE_DIR = next(
    path for path in (Path.cwd(), Path.cwd() / "code")
    if (path / "stag_data.py").is_file()
)
PROJECT_ROOT = CODE_DIR.parent
ZIP_PATH = PROJECT_ROOT / "stag_data" / "classification_lite.zip"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from stag_data import (
    build_continuity_report,
    build_recording_indices,
    build_recording_table,
    build_window_index,
    inspect_mat_variables,
    inspect_zip,
    load_stag_metadata,
    summarize_window_index,
)
from visualization import (
    plot_class_distribution,
    plot_recording_timeline,
    plot_sensor_mask,
    plot_tactile_frame,
    plot_tactile_sequence,
)

print(f"项目根目录：{PROJECT_ROOT}")
print(f"数据压缩包：{ZIP_PATH}")

TypeError: dataclass() got an unexpected keyword argument 'slots'

## 1. 先看 ZIP 和 MAT 文件结构

Lite 压缩包只包含汇总后的 `metadata.mat` 和官方 `readme.html`。代码直接在内存中读取 MAT 文件，不会解压到项目目录。

In [ ]:
display(inspect_zip(ZIP_PATH))
display(inspect_mat_variables(ZIP_PATH))

字段说明：

- `pressure`：全部触觉帧，形状应为 `[135187, 32, 32]`；
- `batchId`、`recordingId`：确定一条连续 recording；
- `objectId`：原始类别编号，0 是 `empty_hand`；
- `splitId`：官方训练/测试划分；
- `hasValidLabel`：当前帧是否有可靠分类标签；
- `isBalanced`：官方单帧平衡子集标志，不能用于重建连续时间轴；
- MAT 中实际字段名是 `isGrasp`，不是部分说明文档中的 `isGrap`。

In [ ]:
metadata = load_stag_metadata(ZIP_PATH)

print(f"总帧数：{metadata.num_frames:,}")
print(f"recording 数：{metadata.num_recordings}")
print(f"原始类别数：{metadata.num_objects}")
print(f"压力形状与类型：{metadata.pressure.shape}, {metadata.pressure.dtype}")
print(f"全局压力范围：{metadata.pressure.min()} ~ {metadata.pressure.max()}")
print(f"有效传感器数：{metadata.sensor_mask.sum()}")
print(f"批次名称：{metadata.batches}")
print(f"划分名称：{metadata.splits}")
print(f"原始类别：{metadata.objects}")

## 2. recording、类别和官方划分

时序任务必须先按 `(batchId, recordingId)` 分组，再在组内排序。下面每一行都代表一条独立 recording，而不是一张压力图。

In [ ]:
recording_table = build_recording_table(metadata)
display(recording_table.head(10))

print("各划分 recording 数：")
display(recording_table.groupby("split").size().rename("recordings"))

print("各类别、各划分 recording 数：")
recording_pivot = pd.crosstab(
    recording_table["object_name"],
    recording_table["split"],
)
display(recording_pivot)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(
    data=recording_table,
    x="object_name",
    y="frame_count",
    hue="split",
    ax=ax,
)
ax.set_title("每条 recording 的原始帧数")
ax.set_xlabel("物体类别")
ax.set_ylabel("帧数")
ax.tick_params(axis="x", rotation=65)
fig.tight_layout()
plt.show()

## 3. 548 点传感器掩码

`32×32=1024` 个矩阵位置中，476 个填充位置在整个数据集中始终等于 510；其余 548 个位置会随触觉压力变化。模型归一化后还会再次应用这个掩码。

In [ ]:
plot_sensor_mask(metadata.sensor_mask)
plt.show()

constant_positions = metadata.pressure.max(axis=0) == metadata.pressure.min(axis=0)
constant_values = metadata.pressure.min(axis=0)[constant_positions]
print(f"恒定位置数：{constant_positions.sum()}")
print(f"恒定位置的唯一数值：{np.unique(constant_values)}")

## 4. 单帧压力与压力分布

In [ ]:
sample_index = int(np.flatnonzero(metadata.has_valid_label)[0])
plot_tactile_frame(
    metadata.pressure[sample_index],
    sensor_mask=metadata.sensor_mask,
    title=f"原始触觉帧 #{sample_index}，类别：{metadata.objects[metadata.object_id[sample_index]]}",
)
plt.show()

# 只统计真实传感器，避免 476 个恒定填充位置主导直方图。
valid_pressure = metadata.pressure[:, metadata.sensor_mask]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid_pressure.ravel(), bins=100, log=True)
ax.axvline(650, color="orange", linestyle="--", label="常用归一化上界 650")
ax.axvline(950, color="red", linestyle="--", label="异常阈值 950")
ax.set_title("真实传感器原始压力分布（纵轴为对数）")
ax.set_xlabel("压力值")
ax.set_ylabel("数量")
ax.legend()
fig.tight_layout()
plt.show()

## 5. 一条 recording 的真实时间线

绿色背景表示 `hasValidLabel=True`。在 26 个真实物体 recording 中，它通常对应可靠接触；`empty_hand` 类需要单独理解，所以正式 26 类任务会过滤它。

In [ ]:
example_row = recording_table.loc[recording_table["object_name"] == "mug"].iloc[0]
example_key = (int(example_row["batch_id"]), int(example_row["recording_id"]))
plot_recording_timeline(metadata, *example_key)
plt.show()

ordered = build_recording_indices(metadata)[example_key]
valid_positions = np.flatnonzero(metadata.has_valid_label[ordered])
sequence_start = max(0, int(valid_positions[len(valid_positions) // 2]) - 8)
sequence_indices = ordered[sequence_start : sequence_start + 16]
plot_tactile_sequence(
    metadata.pressure[sequence_indices],
    timestamps=metadata.timestamp[sequence_indices],
    sensor_mask=metadata.sensor_mask,
    max_frames=8,
)
plt.show()

## 6. 时间断点与异常帧

不能先删除异常帧再把前后两侧连接。正确做法是让异常帧、缺帧和明显时间跳跃成为 chunk 边界。

In [ ]:
continuity = build_continuity_report(metadata)
print("连续性统计总计：")
display(continuity[["structural_breaks", "bad_frames"]].sum().to_frame("count"))
print("存在断点或异常帧的 recording：")
display(
    continuity.loc[
        (continuity["structural_breaks"] > 0) | (continuity["bad_frames"] > 0)
    ].reset_index(drop=True)
)

## 7. 构建连续时序窗口

默认过滤 `empty_hand`，使用 16 帧窗口和 8 帧步长。`interaction` 模式允许窗口包含压力建立/释放阶段，但要求至少一半帧具有可靠标签。

In [ ]:
window_index = build_window_index(
    metadata,
    window_size=16,
    stride=8,
    mode="interaction",
    min_valid_ratio=0.5,
    include_empty_hand=False,
)
display(summarize_window_index(window_index))
display(window_index.manifest.head())

plot_class_distribution(window_index.manifest, window_index.class_names)
plt.show()

## 8. 为什么不能使用 `isBalanced`

`isBalanced` 是为官方单帧实验抽取的离散平衡子集。相邻被选中帧不一定在时间上连续；若先按它过滤，再把剩余帧排列成序列，就会制造不存在的时间邻接。

本项目只把 `isBalanced` 用于探索统计，不让它参与窗口生成。连续窗口只依据 recording、时间、帧号、异常断点与 `hasValidLabel` 比例构建。

In [ ]:
print(f"hasValidLabel=True 帧数：{metadata.has_valid_label.sum():,}")
print(f"isBalanced=True 帧数：{metadata.is_balanced.sum():,}")
print(f"isGrasp=True 帧数：{metadata.is_grasp.sum():,}")
print(f"isTransition=True 帧数：{metadata.is_transition.sum():,}")
print("\n探索完成：本 notebook 没有导入模型，也没有执行训练。")